# WandB LLM Sweep Analysis

This notebook fetches data from the intention-jailbreak-llm-sweep project and visualizes embedding cosine similarity across models.

In [12]:
import wandb
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import json
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

## Configuration

In [13]:
# WandB project details
PROJECT_NAME = "intention-jailbreak-llm-sweep"
ENTITY = os.getenv("WANDB_ENTITY")

# Output directory for figures
FIGS_DIR = Path("../data/imgs")
FIGS_DIR.mkdir(parents=True, exist_ok=True)

## Fetch Data from WandB

In [14]:
# Initialize wandb API
api = wandb.Api()

# Fetch runs from the project
runs = api.runs(f"{ENTITY + '/' if ENTITY else ''}{PROJECT_NAME}")

print(f"Found {len(runs)} runs in project {PROJECT_NAME}")

Found 8 runs in project intention-jailbreak-llm-sweep


In [20]:
# Extract relevant data from runs
data = []

for run in runs:
    # Parse config if it's a JSON string
    if isinstance(run.config, str):
        config = json.loads(run.config)
    else:
        config = run.config
    
    # Get model name from config - handling nested structure from sweep
    model_path = None
    if "_name_or_path" in config:
        if isinstance(config["_name_or_path"], dict):
            model_path = config["_name_or_path"].get("value")
        else:
            model_path = config["_name_or_path"]
    
    if not model_path:
        continue
    
    # Fetch metrics from summary - handle both dict and string JSON
    if hasattr(run.summary, '_json_dict'):
        summary_data = run.summary._json_dict
        if isinstance(summary_data, str):
            summary_dict = json.loads(summary_data)
        else:
            summary_dict = summary_data
    else:
        summary_dict = {}
    
    val_semantic_sim = summary_dict.get("val_semantic_sim", None)
    
    if val_semantic_sim is not None:
        # Strip organization from model name (e.g., "meta-llama/Llama-3.1-8B" -> "Llama-3.1-8B")
        model_name = model_path.split("/")[-1] if "/" in model_path else model_path
        
        data.append({
            "model": model_name,
            "val_semantic_sim": val_semantic_sim,
            "run_name": run.name,
            "run_id": run.id
        })

# Create DataFrame
df = pd.DataFrame(data)

# Capitalize the beginning of model names for better aesthetics
df["model"] = df["model"].apply(lambda x: x[0].upper() + x[1:] if x else x)

print(f"Extracted data for {len(df)} runs with val_semantic_sim")
df

Extracted data for 8 runs with val_semantic_sim


,model,val_semantic_sim,run_name,run_id
0,Qwen3-0.6B,0.715380,fiery-spaceship-20,g8hvwj33
1,Qwen3-4B,0.734821,fresh-salad-26,gc55b40d
2,Qwen3-8B,0.732988,wild-elevator-23,p28ksu1l
3,Gemma-3-4b-it,0.737473,deft-capybara-36,dis0qfs5
4,Gemma-3-12b-it,0.742111,divine-shape-43,riq9iqn9
5,Qwen3-14B,0.741341,fiery-bee-43,so9har4c
6,Llama-3.2-1B-Instruct,0.714917,curious-galaxy-45,b1kts1wp
7,Llama-3.1-8B-Instruct,0.749976,earthy-shadow-46,oa3e7hab


## Visualize Embedding Cosine Similarity

In [22]:
# Sort by semantic similarity in descending order (highest first)
df_sorted = df.sort_values("val_semantic_sim", ascending=False).reset_index(drop=True)

# Create LaTeX table with booktabs
latex_table = df_sorted[["model", "val_semantic_sim"]].copy()
latex_table.columns = ["Model", "Embedding Cosine Similarity"]

# Format the similarity values to 4 decimal places
latex_table["Embedding Cosine Similarity"] = latex_table["Embedding Cosine Similarity"].apply(lambda x: f"{x:.4f}")

# Generate LaTeX code
latex_code = latex_table.to_latex(
    index=False,
    escape=False,
    column_format="lc",
    caption="Embedding Cosine Similarity by Model (LLM Sweep)",
    label="tab:embedding_similarity",
    position="htbp"
)

# Replace default table environment with booktabs commands
latex_code = latex_code.replace(r"\toprule", r"\toprule")
latex_code = latex_code.replace(r"\midrule", r"\midrule")
latex_code = latex_code.replace(r"\bottomrule", r"\bottomrule")

print("LaTeX Table:")
print("=" * 80)
print(latex_code)
print("=" * 80)

# Save to file
output_file = FIGS_DIR / "embedding_cosine_similarity_table.tex"
with open(output_file, "w") as f:
    f.write(latex_code)

print(f"\nTable saved to {output_file}")

# Display the table in pandas format for preview
latex_table

LaTeX Table:
\begin{table}[htbp]
\caption{Embedding Cosine Similarity by Model (LLM Sweep)}
\label{tab:embedding_similarity}
\begin{tabular}{lc}
\toprule
Model & Embedding Cosine Similarity \\
\midrule
Llama-3.1-8B-Instruct & 0.7500 \\
Gemma-3-12b-it & 0.7421 \\
Qwen3-14B & 0.7413 \\
Gemma-3-4b-it & 0.7375 \\
Qwen3-4B & 0.7348 \\
Qwen3-8B & 0.7330 \\
Qwen3-0.6B & 0.7154 \\
Llama-3.2-1B-Instruct & 0.7149 \\
\bottomrule
\end{tabular}
\end{table}


Table saved to ../data/imgs/embedding_cosine_similarity_table.tex


,Model,Embedding Cosine Similarity
0,Llama-3.1-8B-Instruct,0.7500
1,Gemma-3-12b-it,0.7421
2,Qwen3-14B,0.7413
3,Gemma-3-4b-it,0.7375
4,Qwen3-4B,0.7348
5,Qwen3-8B,0.7330
6,Qwen3-0.6B,0.7154
7,Llama-3.2-1B-Instruct,0.7149


## Summary Statistics

In [ ]:
# Display summary statistics
print("Summary Statistics:")
print(f"Mean Embedding Cosine Similarity: {df['val_semantic_sim'].mean():.4f}")
print(f"Median Embedding Cosine Similarity: {df['val_semantic_sim'].median():.4f}")
print(f"Min: {df['val_semantic_sim'].min():.4f} ({df.loc[df['val_semantic_sim'].idxmin(), 'model']})")
print(f"Max: {df['val_semantic_sim'].max():.4f} ({df.loc[df['val_semantic_sim'].idxmax(), 'model']})")

# Display sorted table
df_sorted[["model", "val_semantic_sim"]].rename(columns={"val_semantic_sim": "Embedding Cosine Similarity"})